# Hugging Face + DistilBERT environment spike

## Setup

In [ ]:
import importlib.util, sys

missing = [p for p in ("transformers", "datasets", "torch") if importlib.util.find_spec(p) is None]
if missing:
    !{sys.executable} -m pip install -q {' '.join(missing)}

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import transformers
import datasets
from transformers import AutoTokenizer

## Runtime check

In [ ]:
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)} (CUDA {torch.version.cuda})")
else:
    print("GPU          : not available — enable GPU runtime on Colab/Kaggle")

## Tokenizer

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"vocab size : {tokenizer.vocab_size}")
print(f"max length : {tokenizer.model_max_length}")

## Tokenization sanity check

In [ ]:
TRAIN_CSV = next(p for p in (Path("../data/train.csv"), Path("data/train.csv")) if p.exists())
sample = pd.read_csv(TRAIN_CSV).sample(n=200, random_state=42)

enc = tokenizer(sample["text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")
lengths = enc["attention_mask"].sum(dim=1)
print(f"input_ids shape : {tuple(enc['input_ids'].shape)}")
print(f"token length    : min={lengths.min().item()} median={int(lengths.median())} max={lengths.max().item()}")
print(f"truncated@128   : {(lengths == 128).sum().item()} / {len(lengths)}")